<a href="https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
<a href="https://www.kaggle.com/kernels/welcome?src=https://github.com/unslothai/unsloth/blob/main/studio/unsloth-studio-kaggle.ipynb" target="_blank"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open in Kaggle"/></a>

On **Kaggle**, enable **Internet** and a **GPU** (e.g. Tesla T4), then choose **Run all**. For Google Colab, use [Unsloth_Studio_Colab.ipynb](https://github.com/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb) instead.
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth Studio on your local device, follow [our guide](https://unsloth.ai/docs/new/unsloth-studio/install). Unsloth Studio is licensed [AGPL-3.0](https://github.com/unslothai/unsloth/blob/main/studio/LICENSE.AGPL-3.0).

### Unsloth Studio

Train and run open models with [**Unsloth Studio**](https://unsloth.ai/docs/new/unsloth-studio/start). NEW! Installation should now only take 2 mins!


The Studio backend may bind to a port other than 8888 if Jupyter is already using it; the next cell starts the server first, then opens an **ngrok** tunnel to the actual port automatically.

[Features](https://unsloth.ai/docs/new/unsloth-studio#features) • [Quickstart](https://unsloth.ai/docs/new/unsloth-studio/start) • [Data Recipes](https://unsloth.ai/docs/new/unsloth-studio/data-recipe) • [Studio Chat](https://unsloth.ai/docs/new/unsloth-studio/chat) • [Export](https://unsloth.ai/docs/new/unsloth-studio/export)

<p align="left"><img src="https://github.com/unslothai/unsloth/raw/main/studio/frontend/public/studio%20github%20landscape%20colab%20display.png" width="600"></p>

### Setup: Clone repo and run setup

In [ ]:
!test -d /kaggle/working/unsloth/.git || git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git /kaggle/working/unsloth
%cd /kaggle/working/unsloth
!chmod +x studio/setup.sh && ./studio/setup.sh

### Install pyngrok and start Unsloth Studio (ngrok tunnel)

Get an [ngrok authtoken](https://dashboard.ngrok.com/get-started/your-authtoken), then provide it in **one** of these ways (first match wins):

1. **Notebook variable** — In the next cell, set `NGROK_AUTHTOKEN = "your_token_here"` (or leave `""` to skip).
2. **Kaggle secret** — **Add-ons → Secrets**, name `NGROK_AUTHTOKEN`.
3. **Environment** — `NGROK_AUTHTOKEN` in the process environment.

Avoid committing or sharing notebooks that contain a real token.

In [ ]:
# Optional: paste your ngrok authtoken between the quotes.
# Leave "" to use Kaggle Secrets (name: NGROK_AUTHTOKEN), or the NGROK_AUTHTOKEN environment variable.
# Do not commit a notebook that contains a real token.
NGROK_AUTHTOKEN = ""

In [ ]:
!pip install -q pyngrok

In [ ]:
import logging
import os
import sys
import time
import urllib.request
from pathlib import Path

from pyngrok import ngrok

try:
    sys.stdout.reconfigure(line_buffering=True)
except Exception:
    logging.debug("sys.stdout.reconfigure failed", exc_info=True)

# Repo root: setup cell does `%cd unsloth` under /kaggle/working
REPO_ROOT = Path.cwd()
if REPO_ROOT.name != "unsloth":
    REPO_ROOT = Path("/kaggle/working/unsloth")

BACKEND_DIR = REPO_ROOT / "studio" / "backend"
sys.path.insert(0, str(BACKEND_DIR))
os.chdir(BACKEND_DIR)


def _get_ngrok_token():
    nb = globals().get("NGROK_AUTHTOKEN")
    if nb is not None and str(nb).strip():
        return str(nb).strip()
    try:
        from kaggle_secrets import UserSecretsClient

        t = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
        if t and t.strip():
            return t.strip()
    except Exception:
        logging.debug("Failed to retrieve ngrok token from Kaggle secrets", exc_info=True)
    token = os.environ.get("NGROK_AUTHTOKEN")
    return token.strip() if token else None


token = _get_ngrok_token()
if not token:
    raise RuntimeError(
        "Missing ngrok token. Set NGROK_AUTHTOKEN in the cell above, or add Kaggle secret "
        "NGROK_AUTHTOKEN (Add-ons → Secrets), or set environment variable NGROK_AUTHTOKEN."
    )

ngrok.set_auth_token(token)
ngrok.kill()  # drop stale tunnels when re-running the cell

import run as _run_mod

app = _run_mod.run_server(
    host="0.0.0.0",
    port=8888,
    frontend_path=REPO_ROOT / "studio" / "frontend" / "dist",
    silent=True,
)

actual_port = getattr(app.state, "server_port", None)
if not actual_port:
    raise RuntimeError("Could not determine Studio listening port.")

# ngrok free tier: ask the agent to add this header on forwarded requests when supported.
_NGROK_SKIP_HEADER = {"ngrok-skip-browser-warning": "1"}

try:
    tunnel = ngrok.connect(
        actual_port,
        request_header_add=["ngrok-skip-browser-warning: 1"],
    )
except TypeError:
    # Older pyngrok versions do not support request_header_add
    tunnel = ngrok.connect(actual_port)

public_url = tunnel.public_url

from startup_banner import stdout_supports_color


def _print_ngrok_ready_footer(url: str, port: int) -> None:
    use_color = stdout_supports_color()
    dim = "\033[38;5;245m"
    title = "\033[38;5;150m"
    reset = "\033[0m"

    def style(text: str, code: str) -> str:
        return f"{code}{text}{reset}" if use_color else text

    print(flush=True)
    print(style("🦥 Unsloth Studio is running", title), flush=True)
    print(style("─" * 52, dim), flush=True)
    print("Unsloth Studio is running.", flush=True)
    print(
        f"Listening locally on port {port} "
        "(Jupyter may own 8888; the Studio port is chosen automatically).",
        flush=True,
    )
    print(flush=True)
    print("Open this ngrok URL in your browser — not http://localhost:", flush=True)
    print(url, flush=True)
    print("=" * 60, flush=True)
    print(flush=True)


_print_ngrok_ready_footer(public_url, actual_port)

_health_url = public_url.rstrip("/") + "/api/health"
print("Tunnel check (sends ngrok-skip-browser-warning; any value is valid):", flush=True)
try:
    req = urllib.request.Request(_health_url, headers=_NGROK_SKIP_HEADER)
    with urllib.request.urlopen(req, timeout=60) as resp:
        status = resp.status
        snippet = resp.read(400).decode(errors="replace")
    print(f"  GET /api/health → {status}", flush=True)
    print(f"  {snippet[:400]}", flush=True)
except Exception as exc:
    print(f"  Request failed: {exc}", flush=True)

print(flush=True)
print(
    "Server logs stream below. On ngrok free tier, the first browser visit may show "
    "a warning page -- click Visit Site to continue.",
    flush=True,
)
print(flush=True)

# Short sleeps keep Jupyter’s output stream responsive for uvicorn / background threads.
try:
    while True:
        time.sleep(0.5)
        sys.stdout.flush()
        sys.stderr.flush()
        for lg in (logging.root, logging.getLogger("uvicorn")):
            for h in getattr(lg, "handlers", []) or []:
                try:
                    h.flush()
                except Exception:
                    logging.debug("Failed to flush log handler", exc_info=True)
except KeyboardInterrupt:
    pass
finally:
    print("Closing ngrok tunnel...", flush=True)
    try:
        ngrok.disconnect(public_url)
    except Exception:
        pass
    ngrok.kill()
    shutdown = getattr(getattr(app, "state", None), "trigger_shutdown", None)
    if callable(shutdown):
        try:
            shutdown()
        except Exception:
            logging.debug("Failed to shut down Studio server", exc_info=True)

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook is licensed <a href="https://github.com/unslothai/unsloth/blob/main/studio/LICENSE.AGPL-3.0">AGPL-3.0</a></b>
</div>